In [1]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from functions import *
import igraph as ig

In [ ]:
N,kavg,B,eta = 10000,10,2,0.9
Gset_0,partition = gen_SBM_set(N,kavg,B,eta)
G0nx = graph_Gset(Gset_0)
n_sims = 1

part = nx.community.greedy_modularity_communities(G0nx,cutoff=10) #can use SBM with fixed B = 10 instead
partition_B10 = np.zeros(N).astype('int')
for c,Set in enumerate(part):
    for i in Set:
        partition_B10[i] = c
        
part = nx.community.greedy_modularity_communities(G0nx,cutoff=100) #can use SBM with fixed B = 100 instead
partition_B100 = np.zeros(N).astype('int')
for c,Set in enumerate(part):
    for i in Set:
        partition_B100[i] = c

noises = np.linspace(0,1,20)
noisy_graphs_t1 = [[typeI(Gset_0, eps) for trial in range(n_sims)]for eps in noises]
noisy_graphs_t2 = [[typeII(Gset_0, eps) for trial in range(n_sims)] for eps in noises]
#noisy_graphs_t3 = [typeIII(Gset_0, eps) for eps in noises]

NMI_regs = [np.mean([graphNMI(N,G,Gset_0) for G in samples]) for samples in noisy_graphs_t1]
NMI_DCs = [np.mean([graphDCNMI(G,Gset_0) for G in samples]) for samples in noisy_graphs_t1]
Jacs = [np.mean([jaccard(G,Gset_0) for G in samples]) for samples in noisy_graphs_t1]
meso10 = [np.mean([mesoNMI(G,Gset_0,partition_B10) for G in samples]) for samples in noisy_graphs_t1]
meso100 = [np.mean([mesoNMI(G,Gset_0,partition_B100) for G in samples]) for samples in noisy_graphs_t1]

NMI_regs_err = [3*np.std([graphNMI(N,G,Gset_0) for G in samples])/np.sqrt(len(samples)) for samples in noisy_graphs_t1]
NMI_DCs_err = [3*np.std([graphDCNMI(G,Gset_0) for G in samples])/np.sqrt(len(samples)) for samples in noisy_graphs_t1]
Jacs_err = [3*np.std([jaccard(G,Gset_0) for G in samples])/np.sqrt(len(samples)) for samples in noisy_graphs_t1]
meso10_err = [3*np.std([mesoNMI(G,Gset_0,partition_B10) for G in samples])/np.sqrt(len(samples)) for samples in noisy_graphs_t1]
meso100_err = [3*np.std([mesoNMI(G,Gset_0,partition_B100) for G in samples])/np.sqrt(len(samples)) for samples in noisy_graphs_t1]

plt.errorbar(x=noises,y=NMI_regs,yerr=NMI_regs_err,label='reg')
plt.errorbar(x=noises,y=NMI_DCs,yerr=NMI_DCs_err,label='DC')
plt.errorbar(x=noises,y=Jacs,yerr=Jacs_err,label='Jac')
plt.errorbar(x=noises,y=meso10,yerr=meso10_err,label='meso10')
plt.errorbar(x=noises,y=meso100,yerr=meso100_err,label='meso100')

plt.legend()